# 🌸 ShiroLLM - AI Chatbot (Google Colab Edition)
Jalankan AI Shiro menggunakan GPU gratis di Google Colab (T4 / A100)!

### Fitur Utama:
- 🚀 **Akselerasi GPU CUDA Penuh**: Respon kilat (< 1 detik)
- 🧠 **Model Tercanggih**: Lumimaid-v0.2-8B (Llama-3 fine-tune khusus roleplay & percakapan karakter)
- 💡 **Self-Learning Continuous Memory**: Shiro belajar dan mengingat fakta, janji, emosi, dan preferensi dari setiap percakapan
- 🌐 **Web UI Publik**: Akses antarmuka chat cantik langsung dari browser Anda via Cloudflare Tunnel

## Langkah 1: Cek GPU

In [ ]:
!nvidia-smi

## Langkah 2: Clone Repository

In [ ]:
%cd /content
!rm -rf ShiroLLM-AI-CHAT
!git clone https://github.com/Renn-devBI/ShiroLLM-AI-CHAT.git
%cd /content/ShiroLLM-AI-CHAT

## Langkah 3: Install Dependencies & llama-cpp-python dengan CUDA

In [ ]:
# Install dependencies dasar
!pip install -r requirements.txt
!pip install flask flask-cors huggingface-hub

# Install llama-cpp-python dengan akselerasi CUDA untuk GPU Colab
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

## Langkah 4: Download Model Tercanggih (Lumimaid 8B)

In [ ]:
!python download_model.py

## Langkah 5: Jalankan Shiro-LLMA dengan GPU + Cloudflare Tunnel (Akses Web UI)

In [ ]:
import os
import subprocess
import threading
import time

# Set GPU layers ke -1 (offload semua layer model ke GPU VRAM)
os.environ["N_GPU_LAYERS"] = "-1"

# Download Cloudflared untuk tunnel publik tanpa perlu token/login
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

def start_tunnel():
    time.sleep(3)
    print("\n" + "="*60)
    print("🌐 MEMBUKA LINK WEB UI PUBLIK VIA CLOUDFLARE TUNNEL...")
    print("="*60)
    p = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7474"], 
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "trycloudflare.com" in line:
            print(f"\n👉 LINK WEB UI ANDA: {line.strip()}\n")
            break

threading.Thread(target=start_tunnel, daemon=True).start()

# Jalankan server web Shiro
!python web.py

## (Opsional) Sinkronisasi / Backup Memori ke Google Drive
Jalankan cell ini jika ingin menyimpan ingatan Shiro ke Google Drive agar tidak hilang saat runtime Colab di-restart.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')
backup_dir = '/content/drive/MyDrive/Shiro_Memory_Backup'
os.makedirs(backup_dir, exist_ok=True)

# Backup file ingatan saat ini ke Google Drive
if os.path.exists('ingatan_shiro.json'):
    shutil.copy('ingatan_shiro.json', os.path.join(backup_dir, 'ingatan_shiro.json'))
    print(f"✓ Ingatan Shiro berhasil dibackup ke: {backup_dir}")
